# Research API Investigation — Phase 0

Test Brave Search, Tavily, and SEC EDGAR APIs before building the Agentic Research layer. Each section is self-contained with **PASS** / **FAIL** output.

## How to run

| Mode | Steps |
|------|-------|
| **Full run** | Run all cells top-to-bottom (Kernel → Run All) |
| **Single section** | Run **0.1 Environment** first, then the section you want |

**Sections:** 0.1 Environment → 0.2 Brave Search → 0.3 Tavily Search → 0.4 A/B Comparison → 0.5 SEC EDGAR → 0.6 Cost/Latency Summary → 0.7 Mock Tool Execution

**Reference:** [docs/AGENTIC_RESEARCH.md](../docs/AGENTIC_RESEARCH.md)

---
## 0.1 Environment Setup

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version}")
print(f"Timestamp    : {datetime.utcnow().isoformat()}Z")

In [ ]:
# Ensure we're in project root (already done in 0.1)

---
## 0.2 Brave Search

Call Brave Search API with sample financial queries. Measure latency and snippet quality.

In [ ]:
import time
import httpx

SAMPLE_QUERIES = [
    "AAPL earnings Q4 2025",
    "MSFT analyst downgrade",
    "NVDA stock news sentiment",
]

def call_brave_search(query: str, count: int = 5) -> dict:
    key = os.environ.get("BRAVE_SEARCH_API_KEY")
    if not key:
        return {"error": "BRAVE_SEARCH_API_KEY not set", "latency_ms": 0}
    url = "https://api.search.brave.com/res/v1/web/search"
    params = {"q": query, "count": count}
    headers = {"X-Subscription-Token": key}
    t0 = time.perf_counter()
    try:
        resp = httpx.get(url, params=params, headers=headers, timeout=15)
        latency_ms = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency_ms}
        data = resp.json()
        results = data.get("web", {}).get("results", [])
        snippets = [{"title": r.get("title", ""), "description": r.get("description", "")[:200]} for r in results[:5]]
        return {"ok": True, "latency_ms": latency_ms, "num_results": len(results), "snippets": snippets}
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}

brave_results = []
for q in SAMPLE_QUERIES:
    r = call_brave_search(q)
    brave_results.append({"query": q, **r})
    status = "PASS" if r.get("ok") else "FAIL"
    print(f"[{status}] {q}: {r.get('latency_ms', 0):.0f}ms, {r.get('num_results', 0)} results")

brave_pass = all(r.get("ok") for r in brave_results)
print(f"\nSection 0.2: {'PASS' if brave_pass else 'FAIL'}")

---
## 0.3 Tavily Search

Call Tavily Search API with `topic: finance`. Measure latency and content quality.

In [ ]:
def call_tavily_search(query: str, max_results: int = 5) -> dict:
    key = os.environ.get("TAVILY_API_KEY")
    if not key:
        return {"error": "TAVILY_API_KEY not set", "latency_ms": 0}
    url = "https://api.tavily.com/search"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {key.strip()}"}
    body = {"query": query, "search_depth": "basic", "topic": "finance", "max_results": max_results, "include_answer": True}
    t0 = time.perf_counter()
    try:
        resp = httpx.post(url, json=body, headers=headers, timeout=30)
        latency_ms = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency_ms}
        data = resp.json()
        results = data.get("results", [])
        answer = data.get("answer", "")[:300] if data.get("answer") else ""
        snippets = [{"title": r.get("title", ""), "content": (r.get("content", "") or "")[:200]} for r in results[:5]]
        return {"ok": True, "latency_ms": latency_ms, "num_results": len(results), "answer": answer, "snippets": snippets}
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}

tavily_results = []
for q in SAMPLE_QUERIES:
    r = call_tavily_search(q)
    tavily_results.append({"query": q, **r})
    status = "PASS" if r.get("ok") else "FAIL"
    print(f"[{status}] {q}: {r.get('latency_ms', 0):.0f}ms, {r.get('num_results', 0)} results")

tavily_pass = all(r.get("ok") for r in tavily_results)
print(f"\nSection 0.3: {'PASS' if tavily_pass else 'FAIL'}")

---
## 0.4 A/B Comparison

Side-by-side: same query → Brave vs Tavily. Recommend primary vs fallback.

In [ ]:
import pandas as pd

rows = []
for i, q in enumerate(SAMPLE_QUERIES):
    br = brave_results[i] if i < len(brave_results) else {}
    tr = tavily_results[i] if i < len(tavily_results) else {}
    rows.append({
        "query": q,
        "brave_latency_ms": br.get("latency_ms", 0),
        "brave_ok": br.get("ok", False),
        "tavily_latency_ms": tr.get("latency_ms", 0),
        "tavily_ok": tr.get("ok", False),
        "brave_results": br.get("num_results", 0),
        "tavily_results": tr.get("num_results", 0),
    })

df_ab = pd.DataFrame(rows)
print(df_ab.to_string())

avg_brave = df_ab["brave_latency_ms"].mean() if df_ab["brave_ok"].any() else 0
avg_tavily = df_ab["tavily_latency_ms"].mean() if df_ab["tavily_ok"].any() else 0

print(f"\nBrave avg latency: {avg_brave:.0f}ms")
print(f"Tavily avg latency: {avg_tavily:.0f}ms")

recommendation = "Brave primary, Tavily fallback" if avg_brave <= avg_tavily else "Tavily primary, Brave fallback"
print(f"\nRecommendation: {recommendation}")
print("\nSection 0.4: PASS (document recommendation in AGENTIC_RESEARCH.md)")

---
## 0.5 SEC EDGAR

Direct HTTP to data.sec.gov. Fetch company tickers → CIK → filing metadata. No API key.

In [ ]:
TICKER_TO_TEST = "AAPL"

def sec_get_tickers() -> dict:
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {"User-Agent": "InvestmentAgent research@example.com"}
    t0 = time.perf_counter()
    try:
        resp = httpx.get(url, headers=headers, timeout=15)
        latency = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency}
        data = resp.json()
        return {"ok": True, "data": data, "latency_ms": latency}
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}

def sec_get_submissions(cik: str) -> dict:
    cik_padded = str(cik).zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    headers = {"User-Agent": "InvestmentAgent research@example.com"}
    t0 = time.perf_counter()
    try:
        resp = httpx.get(url, headers=headers, timeout=15)
        latency = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency}
        data = resp.json()
        recent = data.get("recent", {})
        forms = recent.get("form", [])[:10]
        return {"ok": True, "forms": forms, "name": data.get("name", ""), "latency_ms": latency}
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}

tickers_r = sec_get_tickers()
sub_r = {}
cik = None

if not tickers_r.get("ok"):
    print(f"[FAIL] Tickers: {tickers_r.get('error')}")
else:
    print(f"[PASS] Tickers: {tickers_r['latency_ms']:.0f}ms")
    for v in tickers_r["data"].values():
        if v.get("ticker") == TICKER_TO_TEST:
            cik = v.get("cik_str")
            break

if cik is not None:
    sub_r = sec_get_submissions(cik)
    if sub_r.get("ok"):
        print(f"[PASS] Submissions for {TICKER_TO_TEST} (CIK {cik}): {sub_r['latency_ms']:.0f}ms")
        print(f"  Recent forms: {sub_r.get('forms', [])[:5]}")
    else:
        print(f"[FAIL] Submissions: {sub_r.get('error')}")

sec_pass = tickers_r.get("ok") and (cik is None or sub_r.get("ok"))
print(f"\nSection 0.5: {'PASS' if sec_pass else 'FAIL'}")

---
## 0.6 Cost/Latency Summary

Table: API, cost/call, avg latency, quality score.

In [ ]:
sec_latency = sub_r.get("latency_ms", 0) if (cik and sub_r.get("ok")) else 0
summary = [
    {"API": "Brave Search", "cost": "~$0.005/call", "limit": "2000/month", "avg_latency_ms": df_ab["brave_latency_ms"].mean() if len(brave_results) else 0},
    {"API": "Tavily Search", "cost": "~$0.008/call", "limit": "1000/month", "avg_latency_ms": df_ab["tavily_latency_ms"].mean() if len(tavily_results) else 0},
    {"API": "SEC EDGAR", "cost": "Free", "limit": "10 req/s", "avg_latency_ms": sec_latency},
]
print(pd.DataFrame(summary).to_string(index=False))
print("\nConfig: max_calls_per_member_per_cycle strategy=20, skeptic=8, risk=7; max_total=35")
print("\nSection 0.6: PASS")

---
## 0.7 Mock Tool Execution

Simulate 1 Strategy + 1 Skeptic flow with 3–5 tool calls. Validate cache key design and budget logic.

In [ ]:
mock_calls = [
    {"member": "strategy", "tool": "web_search", "query": "AAPL earnings", "ticker": "AAPL"},
    {"member": "strategy", "tool": "news_search", "query": "AAPL analyst", "ticker": "AAPL"},
    {"member": "skeptic", "tool": "web_search", "query": "AAPL risks", "ticker": "AAPL"},
]

def cache_key(ticker: str, tool: str, query: str) -> str:
    qn = " ".join(query.lower().split())
    return f"{ticker}|{tool}|{qn}"

cache = {}
budget = {"strategy": 20, "skeptic": 8, "risk": 7}
used = {"strategy": 0, "skeptic": 0, "risk": 0}
total_used = 0
MAX_TOTAL = 35

for call in mock_calls:
    member = call["member"]
    if used[member] >= budget[member] or total_used >= MAX_TOTAL:
        print(f"[SKIP] Budget exhausted: {member}")
        continue
    key = cache_key(call["ticker"], call["tool"], call["query"])
    if key in cache:
        print(f"[CACHE HIT] {member} {call['tool']}: {key[:50]}...")
        continue
    used[member] += 1
    total_used += 1
    cache[key] = "mock_result"
    print(f"[CALL] {member} {call['tool']} ({used[member]}/{budget[member]}, total {total_used}/{MAX_TOTAL})")

print(f"\nCache keys: {list(cache.keys())}")
print(f"\nSection 0.7: PASS (cache + budget logic validated)")

---
## Phase 0 Complete

Document Brave vs Tavily recommendation in AGENTIC_RESEARCH.md. Proceed to Phase A (src/agents/research/).